# CMI Problematic Internet Use — Handcrafted Autocorrelation Baseline & `SequenceExtractor`

This notebook implements a **100% handcrafted, zero-ML baseline** for the **Child Mind Institute — Problematic Internet Use** competition (metric: **Quadratic Weighted Kappa** on ordinal `sii` in `{0, 1, 2, 3}`), structured in the style of `multi-branch_version_3.ipynb`:
- **Data Augmentation (`tsaug`)**: Time-warping, cropping, multiplicative drift, jitter, additive noise, time-shifting, temporal masking, and channel dropout via `TSAugmentor`.
- **Feature Extraction (`SequenceExtractor`)**: Modular multi-library extraction across `tsfel`, `tsfresh`, `tsflex`, `pycatch22` / `sktime`, `seglearn`, `statsmodels`, `scipy.signal`, `PyWavelets`, and handcrafted autocorrelation, circadian (`OpenStrap/analytics`, `cr7_kimmitch`), and sleep-architecture (`mad-lab-fau/sleep_analysis`) descriptors with leak-free missing-data imputation.
- **Handcrafted Estimator (`HandcraftedSequenceBaselineEstimator`)**: Deterministic domain scoring (`autocorr_spectral_rule`, `weighted_z_index`, `multiplicative_hazard`) with ordinal QWK threshold calibration.
- **Separate Search Spaces**: Edit `GRID_PARAM_SPACE` or `BAYESIAN_PARAM_SPACE` in their own dedicated cells under `# Params` with zero `.update()` calls.


In [ ]:
import os
import sys
import json
import warnings
import importlib
from pathlib import Path
from datetime import datetime

_py_env = Path(sys.executable).parent
_lib_bin = _py_env / 'Library' / 'bin'
if _lib_bin.is_dir() and str(_lib_bin) not in os.environ.get('PATH', ''):
    os.environ['PATH'] = str(_lib_bin) + os.pathsep + str(_py_env) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

warnings.filterwarnings('ignore')

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir).lower() == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

for _folder in ('Src', 'src'):
    _src_path = os.path.join(workspace_root, _folder)
    if os.path.isdir(_src_path) and _src_path not in sys.path:
        sys.path.insert(0, _src_path)
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Robust Kaggle path routing (handles nested folders and symlinks from zip uploads)
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        dirs[:] = [d for d in dirs if not d.endswith('.parquet') and not d.startswith('id=')]
        if 'baseline_utils.py' in files or 'base_utils.py' in files:
            if root not in sys.path:
                sys.path.insert(0, root)

from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import baseline_utils
import base_utils
importlib.reload(baseline_utils)
importlib.reload(base_utils)

from base_utils import (
    DEFAULT_SEQUENCE_EXTRACTOR_PARAMS,
    ECat,
    EFloat,
    EInt,
    GASearchCV,
    HandcraftedSequenceBaselineEstimator,
    HierarchicalBayesSearchCV,
    HierarchicalGASearchCV,
    NoopActigraphyOptions,
    SequenceExtractor,
    TSAugmentor,
    build_quick_results_df,
    candidate_accuracy_pct,
    candidate_score,
    clear_actigraphy_caches,
    compute_permutation_importance,
    deduplicate_param_grid,
    evaluate_holdout,
    export_submission,
    find_data_root,
    library_status,
    load_cmi_dataset,
    prepare_bayesian_space,
    prepare_evolutionary_space,
    refit_valid_candidate,
    run_validation_searches,
)

print('Imports loaded successfully')
print('Library availability:', library_status())


# Setup

In [ ]:
experiment_name = 'cmi_piu_handcrafted_baseline'
TARGET_COL = 'sii'

# Set True for a quick sampled smoke test (37 subjects); set False for full train.csv
use_sample_data = False
sample_max_subjects = 37
save_results = not use_sample_data
keep_history = False

search_mode = 'bayes'  # Choose 'bayes', 'evo', or 'grid'
random_state = 42
n_splits = 1           # 1 -> StratifiedShuffleSplit; >=2 -> StratifiedKFold

train_size = 0.8
n_iter = 37

# Evolutionary search controls (active when search_mode == 'evo')
population_size = 20
generations = 4
tournament_size = 3
elitism = 2
crossover_probability = 0.7
mutation_probability = 0.25

# Permutation importance toggle
run_permutation_importance = True

# Leakage-safe two-tier actigraphy cache controls
enable_cache = True
enable_feature_cache = True
cache_dir = None             # Set e.g. '/kaggle/working/act_cache' for disk-backed caching

INVALID_CANDIDATE_POLICY = 'penalize'
INVALID_CANDIDATE_SCORE = -1.0
verbose = 1
error_score = -1.0
n_jobs = 1
refit = refit_valid_candidate

search_mode = str(search_mode).strip().lower()
if search_mode not in ('bayes', 'evo', 'grid'):
    raise ValueError("search_mode must be 'bayes', 'evo', or 'grid'")

results_dir = (
    Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
    if save_results
    else None
)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if n_splits <= 1:
    cv_object = StratifiedShuffleSplit(
        n_splits=1,
        test_size=(1.0 - train_size),
        random_state=random_state,
    )
else:
    cv_object = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

VALIDATION_PARAM_SPACE = {
    'validation_mode': ['stratified_subject'],
}


# Prep

In [ ]:
data_root = find_data_root()
train_df = load_cmi_dataset(
    data_root=data_root,
    split='train',
    dropna_target=True,
    max_subjects=(sample_max_subjects if use_sample_data else None),
    random_state=random_state,
)
print(f'Using {data_root}: {len(train_df)} labeled subjects (actigraphy={int(train_df["has_actigraphy"].sum())})')


In [ ]:
experiment_df = train_df
del train_df
print('Available subjects:', len(experiment_df))


In [ ]:
extractor_params = {
    **DEFAULT_SEQUENCE_EXTRACTOR_PARAMS,
    'enable_cache': enable_cache,
    'enable_feature_cache': enable_feature_cache,
    'cache_dir': cache_dir,
}
extractor = SequenceExtractor(
    noop=NoopActigraphyOptions(),
    **extractor_params,
)

clf = HandcraftedSequenceBaselineEstimator(
    extractor=extractor,
    augmentor=TSAugmentor(prob=0.0, per_aug_prob=0.5, random_state=random_state),
    scoring_rule='autocorr_spectral_rule',
    primary_target=TARGET_COL,
    threshold_calibration='optimize_qwk',
    invalid_candidate_policy=INVALID_CANDIDATE_POLICY,
    invalid_candidate_score=INVALID_CANDIDATE_SCORE,
    verbose=verbose,
    random_state=random_state,
)
pipeline = Pipeline([('estimator', clf)])


# Params

Search spaces are organised into separate, self-contained cells (`GRID_PARAM_SPACE`, `BAYESIAN_PARAM_SPACE`, and `EVOLUTIONARY_PARAM_SPACE`) across 8 ordered sections with zero `.update()` calls.


## Grid

In [ ]:
GRID_PARAM_SPACE = {
    # ----------------------------------------------------------
    # 1. AUGMENTOR (tsaug & Modality Dropout)
    # ----------------------------------------------------------
    'estimator__augmentor__prob': [0.0],
    'estimator__augmentor__per_aug_prob': [0.5],
    'estimator__augmentor__augmentations': [None],
    'estimator__augmentor__n_copies': [1],
    'estimator__augmentor__jitter_sigma': [0.0],
    'estimator__augmentor__noise_std': [0.0],
    'estimator__augmentor__scaling_sigma': [0.0],
    'estimator__augmentor__warp_sigma': [0.0],
    'estimator__augmentor__warp_num_knots': [4],
    'estimator__augmentor__time_warp_max_speed_ratio': [1.0],
    'estimator__augmentor__time_warp_n_speed_change': [3],
    'estimator__augmentor__time_shift_frac': [0.0],
    'estimator__augmentor__crop_frac_range': [(1.0, 1.0)],
    'estimator__augmentor__temporal_num_masks': [0],
    'estimator__augmentor__temporal_mask_frac': [0.0],
    'estimator__augmentor__temporal_mask_fill': ['ffill'],
    'estimator__augmentor__channel_drop_prob': [0.0],
    'estimator__augmentor__sensor_drop_prob': [0.0],
    'estimator__augmentor__sensor_drop_joint_prob': [0.0],
    'estimator__augmentor__random_state': [random_state],

    # ----------------------------------------------------------
    # 2. EXTRACTOR — Cleaning, Resampling, Gap Imputation & Sequence Quality
    # ----------------------------------------------------------
    'estimator__extractor__bin_minutes': [15],
    'estimator__extractor__resample_method': ['anti_alias_mean'],
    'estimator__extractor__non_wear_policy': ['mask', 'impute_diurnal'],
    'estimator__extractor__interp_mode': ['bounded_linear', 'diurnal_median'],
    'estimator__extractor__min_bin_wear_frac': [0.25],
    'estimator__extractor__max_gap_bins': [4],
    'estimator__extractor__impute_long_gaps': [False],
    'estimator__extractor__min_valid_wear_days': [0.0],
    'estimator__extractor__maxlen_days': [None],
    'estimator__extractor__sequence_crop_mode': ['none'],
    'estimator__extractor__window_cap_policy': ['uniform'],
    'estimator__extractor__final_window_mode': ['overlap'],
    'estimator__extractor__filter_problematic_sequences': [False],
    'estimator__extractor__ideal_skew_threshold': [3.5],
    'estimator__extractor__problematic_features_threshold': [3],
    'estimator__extractor__filter_method': ['none', 'iqr', 'zscore'],
    'estimator__extractor__filter_threshold': [3.0],
    'estimator__extractor__max_outlier_features': [3],
    'estimator__extractor__clip_enmo': [4.0],

    # ----------------------------------------------------------
    # 3. EXTRACTOR — Gravity Removal, Kalman Filtering & Kinematic Branches
    # ----------------------------------------------------------
    'estimator__extractor__linear_acc_mode': ['rolling_mean'],
    'estimator__extractor__baseline_window_bins': [16],
    'estimator__extractor__highpass_hz': [0.00015],
    'estimator__extractor__motion_filter_mode': ['kalman'],
    'estimator__extractor__kalman_process_noise': [1e-3],
    'estimator__extractor__kalman_measurement_noise': [1e-2],
    'estimator__extractor__use_dead_reckoning': [True],
    'estimator__extractor__dead_reckoning_detrend': [True],
    'estimator__extractor__acc_modes': ['raw|smoothed|jerk|velocity|linear|kalman'],
    'estimator__extractor__dc_offset_bins': [2],
    'estimator__extractor__smooth_window_bins': [4],
    'estimator__extractor__smooth_alpha': [None],
    'estimator__extractor__sedentary_threshold_g': [0.025],

    # ----------------------------------------------------------
    # 4. EXTRACTOR — Autocorrelation, Lags & Leads, Circadian & Sleep Architecture
    # ----------------------------------------------------------
    'estimator__extractor__acf_detrend': ['linear'],
    'estimator__extractor__acf_bias': ['unbiased'],
    'estimator__extractor__short_lags_h': [(0.25, 0.5, 1.0, 2.0)],
    'estimator__extractor__period_band_hours': [(18.0, 30.0)],
    'estimator__extractor__fundamental_ratio': [0.85],
    'estimator__extractor__min_acf_strength': [0.08],
    'estimator__extractor__include_lags': ['none', True],
    'estimator__extractor__include_leads': ['none', True],
    'estimator__extractor__include_rolling': ['none', True],
    'estimator__extractor__lag_lead_hours': ['none', (0.5, 1.0, 2.0, 4.0)],
    'estimator__extractor__rolling_window_hours': ['none', (1.0, 3.0, 6.0)],
    'estimator__extractor__lag_lead_channels': ['none', 'log_enmo|light|anglez'],

    # ----------------------------------------------------------
    # 5. EXTRACTOR — Time-Frequency (STFT, Welch Moments, CWT Cone/Ridges, DWT)
    # ----------------------------------------------------------
    'estimator__extractor__stft_window_days': [2.0],
    'estimator__extractor__stft_overlap_ratio': [0.5],
    'estimator__extractor__stft_window_type': ['hann'],
    'estimator__extractor__stft_scaling': ['psd'],
    'estimator__extractor__stft_detrend': ['constant'],
    'estimator__extractor__spectral_moment_max_peaks': [6],
    'estimator__extractor__cwt_wavelet': ['morl'],
    'estimator__extractor__cwt_n_scales': [16],
    'estimator__extractor__cwt_min_period_h': [2.0],
    'estimator__extractor__cwt_max_period_h': [36.0],
    'estimator__extractor__dwt_wavelet': ['db4'],
    'estimator__extractor__dwt_level': [4],

    # ----------------------------------------------------------
    # 6. EXTRACTOR — Open-Source Engines & Feature Selection
    # ----------------------------------------------------------
    'estimator__extractor__seglearn_window_days': [1.0],
    'estimator__extractor__seglearn_overlap': [0.5],
    'estimator__extractor__seglearn_feature_set': ['safe_all'],
    'estimator__extractor__catch22_catch24': [True],
    'estimator__extractor__catch22_channels': ['log_enmo|dyn_amp'],
    'estimator__extractor__static_backend': ['tsfel+tsfresh', 'none'],
    'estimator__extractor__tsfel_domains': ['statistical|temporal|spectral'],
    'estimator__extractor__tsfel_features': [None],
    'estimator__extractor__tsfresh_profile': ['minimal'],
    'estimator__extractor__static_channels': ['log_enmo|dyn_amp|anglez|light|lin_enmo|enmo_kalman'],
    'estimator__extractor__tsflex_windows': [('2h', '6h', '12h')],
    'estimator__extractor__tsflex_strides': ['2h'],
    'estimator__extractor__tsflex_channels': ['log_enmo|dyn_amp'],
    'estimator__extractor__feature_selection_method': ['none', 'correlation', 'f_classif'],
    'estimator__extractor__static_top_k': [None, 20],
    'estimator__extractor__feature_selection_percentile': [None],
    'estimator__extractor__feature_families': [
        'autocorr|lag_lead|circadian|sleep_arch|motion|spectral|stft|wavelet|seglearn|catch22|tsfel|tsfresh|tsflex|prsa_cusum|branches|compliance',
    ],

    # ----------------------------------------------------------
    # 7. EXTRACTOR — NOOP Physiology & Tabular Missingness / Imputation
    # ----------------------------------------------------------
    'estimator__extractor__noop__motion_stats': ['enmo_ac_rms|motion_cv|posture_var|jerk_floor|dog_var|baevsky_si'],
    'estimator__extractor__noop__periodicity': ['acf_sublag_fundamental|cosinor'],
    'estimator__extractor__noop__spectral_stats': ['peakedness|spectral_entropy|peak_period|parseval_ratio'],
    'estimator__extractor__noop__quality': ['basic'],
    'estimator__extractor__noop__power_branches': ['enmo|light'],
    'estimator__extractor__include_tabular_engineered': [True],
    'estimator__extractor__harmonize_instruments': [True],
    'estimator__extractor__missing_feature_strategy': ['keep_nan', 'group_age_median'],
    'estimator__extractor__add_missing_indicators': [True],
    'estimator__extractor__impute_knn_neighbors': [5],
    'estimator__extractor__impute_max_iter': [10],

    # ----------------------------------------------------------
    # 8. HANDCRAFTED ESTIMATOR — Domain Rules, Weights & Threshold Calibration
    # ----------------------------------------------------------
    'estimator__scoring_rule': ['autocorr_spectral_rule', 'multiplicative_hazard'],
    'estimator__w_lag_lead_coupling': [0.25],
    'estimator__threshold_calibration': ['optimize_qwk'],
    'estimator__random_state': [random_state],
}


## Bayesian

In [ ]:
# BAYESIAN SEARCH ONLY
BAYESIAN_PARAM_SPACE = dict()
if SKOPT_AVAILABLE:
    tsfel_minimal = {
        'statistical': {
            'Mean': {'use': 'yes'},
            'Variance': {'use': 'yes'},
            'Max': {'use': 'yes'},
            'Min': {'use': 'yes'},
        }
    }
    tsfel_kinematic = {
        'statistical': {
            'Mean': {'use': 'yes'},
            'Variance': {'use': 'yes'},
            'Root mean square': {'use': 'yes'},
        },
        'temporal': {
            'Peak to peak distance': {'use': 'yes'},
            'Zero crossing rate': {'use': 'yes'},
            'Slope': {'use': 'yes'},
        },
    }

    tsfel_entropy = {
        'statistical': {
            'Entropy': {'use': 'yes'},
        },
        'temporal': {
            'Slope': {'use': 'yes'},
            'Zero crossing rate': {'use': 'yes'},
        },
        'spectral': {
            'Spectral entropy': {'use': 'yes'},
            'Fundamental frequency': {'use': 'yes'},
        },
    }

    BAYESIAN_PARAM_SPACE = {
        # ----------------------------------------------------------
        # 1. AUGMENTOR (tsaug & Modality Dropout — 0.0 first so Iteration 0 starts clean, up to extreme limits)
        # ----------------------------------------------------------
        'estimator__augmentor__prob': Categorical([0.0, 0.25, 0.5, 0.75, 1.0]),
        'estimator__augmentor__per_aug_prob': Categorical([0.0, 0.25, 0.35, 0.5, 0.75, 1.0]),
        'estimator__augmentor__augmentations': Categorical([
            'none',
            'None',
            'jitter|noise|scaling',
            'jitter|noise|scaling|magnitude_warp|time_warp',
            'time_warp|time_shift|crop|temporal_mask|channel_drop',
            'jitter|noise|scaling|magnitude_warp|time_warp|time_shift|crop|temporal_mask|channel_drop',
        ]),
        'estimator__augmentor__n_copies': Categorical([0, 1, 2, 3, 4]),
        'estimator__augmentor__jitter_sigma': Categorical([0.0, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0]),
        'estimator__augmentor__noise_std': Categorical([0.0, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0]),
        'estimator__augmentor__scaling_sigma': Categorical([0.0, 0.03, 0.05, 0.10, 0.25, 0.50, 1.0]),
        'estimator__augmentor__warp_sigma': Categorical([0.0, 0.03, 0.05, 0.10, 0.25, 0.50, 1.0]),
        'estimator__augmentor__warp_num_knots': Categorical([0, 2, 4, 6, 8, 12]),
        'estimator__augmentor__time_warp_max_speed_ratio': Categorical([1.0, 1.15, 1.25, 1.5, 2.0, 3.0, 5.0]),
        'estimator__augmentor__time_warp_n_speed_change': Categorical([0, 2, 3, 5, 8, 12, 20]),
        'estimator__augmentor__time_shift_frac': Categorical([0.0, 0.02, 0.05, 0.10, 0.25, 0.50, 0.75, 0.95]),
        'estimator__augmentor__crop_frac_range': Categorical([
            json.dumps([1.0, 1.0]),
            json.dumps([0.85, 1.0]),
            json.dumps([0.75, 0.95]),
            json.dumps([0.5, 0.9]),
            json.dumps([0.4, 0.8]),
            json.dumps([0.1, 0.4]),
        ]),
        'estimator__augmentor__temporal_num_masks': Categorical([0, 1, 2, 4, 8, 15]),
        'estimator__augmentor__temporal_mask_frac': Categorical([0.0, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90]),
        'estimator__augmentor__temporal_mask_fill': Categorical(['none', 'ffill', 'bfill', 'mean', 0.0]),
        'estimator__augmentor__channel_drop_prob': Categorical([0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75]),
        'estimator__augmentor__sensor_drop_prob': Categorical([0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75]),
        'estimator__augmentor__sensor_drop_joint_prob': Categorical([0.0, 0.05, 0.15, 0.20, 0.35, 0.50, 0.75]),
        'estimator__augmentor__random_state': Categorical([random_state]),

        # ----------------------------------------------------------
        # 2. EXTRACTOR — Cleaning, Resampling, Gap Imputation & Sequence Quality
        # ----------------------------------------------------------
        'estimator__extractor__bin_minutes': Categorical([5, 10, 15, 20, 30, 60]),
        'estimator__extractor__resample_method': Categorical(['none', 'anti_alias_mean', 'poly', 'mean']),
        'estimator__extractor__non_wear_policy': Categorical(['none', 'mask', 'impute_zero', 'impute_diurnal', 'keep_all']),
        'estimator__extractor__interp_mode': Categorical([
            'none', 'bounded_linear', 'bounded_pchip', 'bounded_cubic', 'bounded_ffill', 'diurnal_median', 'zero',
        ]),
        'estimator__extractor__min_bin_wear_frac': Categorical([0.0, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75]),
        'estimator__extractor__max_gap_bins': Categorical([0, 1, 2, 4, 8, 12, 24, 48]),
        'estimator__extractor__impute_long_gaps': Categorical([False, True]),
        'estimator__extractor__min_valid_wear_days': Categorical([0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]),
        'estimator__extractor__maxlen_days': Categorical(['None', 3.0, 5.0, 7.0, 14.0, 21.0, 28.0]),
        'estimator__extractor__sequence_crop_mode': Categorical(['none', 'pad_trunc', 'tsaug_crop', 'head', 'tail', 'center']),
        'estimator__extractor__window_cap_policy': Categorical(['uniform', 'first', 'last']),
        'estimator__extractor__final_window_mode': Categorical(['overlap', 'pad', 'drop']),
        'estimator__extractor__filter_problematic_sequences': Categorical([False, True]),
        'estimator__extractor__ideal_skew_threshold': Real(1.5, 6.0),
        'estimator__extractor__problematic_features_threshold': Integer(1, 6),
        'estimator__extractor__filter_method': Categorical(['none', 'iqr', 'zscore']),
        'estimator__extractor__filter_threshold': Real(1.5, 5.0),
        'estimator__extractor__max_outlier_features': Integer(1, 8),
        'estimator__extractor__clip_enmo': Categorical([0.0, 1.5, 2.5, 4.0, 6.0, 10.0]),

        # ----------------------------------------------------------
        # 3. EXTRACTOR — Gravity Removal, Kalman Filtering & Kinematic Branches
        # ----------------------------------------------------------
        'estimator__extractor__linear_acc_mode': Categorical(['none', 'rolling_mean', 'highpass_sos']),
        'estimator__extractor__baseline_window_bins': Categorical([0, 4, 8, 16, 32, 64]),
        'estimator__extractor__highpass_hz': Categorical([0.00005, 0.00010, 0.00015, 0.00020, 0.00025]),
        'estimator__extractor__motion_filter_mode': Categorical(['none', 'kalman']),
        'estimator__extractor__kalman_process_noise': Real(1e-5, 5e-2, prior='log-uniform'),
        'estimator__extractor__kalman_measurement_noise': Real(1e-4, 1e-1, prior='log-uniform'),
        'estimator__extractor__use_dead_reckoning': Categorical([False, True]),
        'estimator__extractor__dead_reckoning_detrend': Categorical([False, True]),
        'estimator__extractor__acc_modes': Categorical([
            'raw',
            'raw|smoothed|jerk',
            'raw|smoothed|jerk|velocity|linear|kalman',
            'raw|smoothed|jerk|velocity|freq_velocity|freq_displacement|linear|kalman',
        ]),
        'estimator__extractor__dc_offset_bins': Integer(1, 6),
        'estimator__extractor__smooth_window_bins': Categorical([0, 2, 4, 8, 12, 16]),
        'estimator__extractor__smooth_alpha': Categorical(['None', 0.0, 0.15, 0.25, 0.5, 0.75]),
        'estimator__extractor__sedentary_threshold_g': Categorical([0.010, 0.015, 0.020, 0.025, 0.035, 0.050]),

        # ----------------------------------------------------------
        # 4. EXTRACTOR — Autocorrelation, Lags & Leads, Circadian & Sleep Architecture
        # ----------------------------------------------------------
        'estimator__extractor__acf_detrend': Categorical(['none', 'linear', 'constant']),
        'estimator__extractor__acf_bias': Categorical(['unbiased', 'biased']),
        'estimator__extractor__short_lags_h': Categorical([
            json.dumps([0.25, 0.5, 1.0, 2.0]),
            json.dumps([0.5, 1.0, 2.0, 4.0]),
            json.dumps([0.25, 0.5, 1.0, 2.0, 4.0, 6.0]),
            json.dumps([1.0, 2.0, 4.0, 8.0, 12.0]),
        ]),
        'estimator__extractor__period_band_hours': Categorical([
            json.dumps([16.0, 32.0]),
            json.dumps([18.0, 30.0]),
            json.dumps([20.0, 28.0]),
            json.dumps([22.0, 26.0]),
        ]),
        'estimator__extractor__fundamental_ratio': Categorical([0.65, 0.75, 0.80, 0.85, 0.90, 0.95]),
        'estimator__extractor__min_acf_strength': Categorical([0.0, 0.03, 0.05, 0.08, 0.12, 0.18]),
        'estimator__extractor__include_lags': Categorical(['none', False, True]),
        'estimator__extractor__include_leads': Categorical(['none', False, True]),
        'estimator__extractor__include_rolling': Categorical(['none', False, True]),
        'estimator__extractor__lag_lead_hours': Categorical([
            'none',
            json.dumps([0.25, 0.5, 1.0]),
            json.dumps([0.5, 1.0, 2.0]),
            json.dumps([0.5, 1.0, 2.0, 4.0]),
            json.dumps([0.5, 1.0, 2.0, 4.0, 6.0, 12.0]),
        ]),
        'estimator__extractor__rolling_window_hours': Categorical([
            'none',
            json.dumps([0.5, 1.5, 3.0]),
            json.dumps([1.0, 3.0]),
            json.dumps([1.0, 3.0, 6.0]),
            json.dumps([1.0, 3.0, 6.0, 12.0, 24.0]),
        ]),
        'estimator__extractor__lag_lead_channels': Categorical([
            'none',
            'log_enmo|light',
            'log_enmo|light|anglez',
            'log_enmo|enmo|dyn_amp|light|anglez|lin_enmo',
        ]),

        # ----------------------------------------------------------
        # 5. EXTRACTOR — Time-Frequency (STFT, Welch Moments, CWT Cone/Ridges, DWT)
        # ----------------------------------------------------------
        'estimator__extractor__stft_window_days': Categorical([1.0, 1.5, 2.0, 3.0, 4.0]),
        'estimator__extractor__stft_overlap_ratio': Categorical([0.0, 0.25, 0.5, 0.75, 0.85]),
        'estimator__extractor__stft_window_type': Categorical(['none', 'hann', 'hamming', 'blackman']),
        'estimator__extractor__stft_scaling': Categorical(['none', 'psd', 'spectrum']),
        'estimator__extractor__stft_detrend': Categorical(['constant', 'linear', 'none']),
        'estimator__extractor__spectral_moment_max_peaks': Integer(2, 12),
        'estimator__extractor__cwt_wavelet': Categorical(['none', 'morl', 'mexh', 'cmor1.5-1.0']),
        'estimator__extractor__cwt_n_scales': Categorical([0, 8, 12, 16, 24, 32]),
        'estimator__extractor__cwt_min_period_h': Categorical([1.0, 1.5, 2.0, 3.0, 4.0]),
        'estimator__extractor__cwt_max_period_h': Categorical([24.0, 30.0, 36.0, 48.0, 72.0]),
        'estimator__extractor__dwt_wavelet': Categorical(['none', 'db4', 'sym4', 'coif1', 'haar']),
        'estimator__extractor__dwt_level': Categorical([0, 2, 3, 4, 5, 6]),

        # ----------------------------------------------------------
        # 6. EXTRACTOR — Open-Source Engines & Feature Selection
        # ----------------------------------------------------------
        'estimator__extractor__seglearn_window_days': Categorical([0.5, 1.0, 2.0, 3.0, 4.0]),
        'estimator__extractor__seglearn_overlap': Categorical([0.0, 0.25, 0.5, 0.75]),
        'estimator__extractor__seglearn_feature_set': Categorical(['none', 'safe_all', 'base', 'emg_kinematic']),
        'estimator__extractor__catch22_catch24': Categorical([False, True]),
        'estimator__extractor__catch22_channels': Categorical([
            'none',
            'log_enmo',
            'log_enmo|dyn_amp',
            'log_enmo|dyn_amp|anglez',
            'log_enmo|dyn_amp|anglez|light',
        ]),
        'estimator__extractor__static_backend': Categorical(['none', 'tsfel+tsfresh', 'tsfel', 'tsfresh']),
        'estimator__extractor__tsfel_domains': Categorical([
            'none', 'statistical', 'temporal', 'spectral', 'statistical|temporal', 'statistical|temporal|spectral',
        ]),
        'estimator__extractor__tsfel_features': Categorical([
            'None',
            'none',
            json.dumps(tsfel_minimal, sort_keys=True),
            json.dumps(tsfel_kinematic, sort_keys=True),
            json.dumps(tsfel_entropy, sort_keys=True),
        ]),
        'estimator__extractor__tsfresh_profile': Categorical(['none', 'minimal', 'efficient']),
        'estimator__extractor__static_channels': Categorical([
            'none',
            'log_enmo|dyn_amp',
            'log_enmo|dyn_amp|anglez|light',
            'log_enmo|dyn_amp|anglez|light|lin_enmo|enmo_kalman',
        ]),
        'estimator__extractor__tsflex_windows': Categorical([
            'none',
            json.dumps(['1h', '3h']),
            json.dumps(['2h', '6h']),
            json.dumps(['2h', '6h', '12h']),
            json.dumps(['2h', '6h', '12h', '24h']),
        ]),
        'estimator__extractor__tsflex_strides': Categorical(['30m', '1h', '2h', '4h', '6h']),
        'estimator__extractor__tsflex_channels': Categorical([
            'none', 'log_enmo|dyn_amp', 'log_enmo|dyn_amp|light', 'log_enmo|dyn_amp|light|anglez',
        ]),
        'estimator__extractor__feature_selection_method': Categorical(['none', 'correlation', 'f_classif']),
        'estimator__extractor__static_top_k': Categorical(['None', 8, 12, 24, 48, 96, 128]),
        'estimator__extractor__feature_selection_percentile': Categorical(['None', 10.0, 25.0, 50.0, 75.0, 90.0]),
        'estimator__extractor__feature_families': Categorical([
            'none',
            'autocorr|lag_lead|circadian|sleep_arch|compliance',
            'autocorr|lag_lead|circadian|sleep_arch|motion|spectral|stft|wavelet|prsa_cusum|branches|compliance',
            'autocorr|lag_lead|circadian|sleep_arch|motion|spectral|stft|wavelet|seglearn|catch22|tsfel|tsfresh|tsflex|prsa_cusum|branches|compliance',
        ]),

        # ----------------------------------------------------------
        # 7. EXTRACTOR — NOOP Physiology & Tabular Missingness / Imputation
        # ----------------------------------------------------------
        'estimator__extractor__noop__motion_stats': Categorical([
            'none',
            'enmo_ac_rms|motion_cv',
            'enmo_ac_rms|motion_cv|posture_var|jerk_floor',
            'enmo_ac_rms|motion_cv|posture_var|jerk_floor|dog_var|baevsky_si',
        ]),
        'estimator__extractor__noop__periodicity': Categorical([
            'none', 'acf', 'acf_harmonic_checked', 'acf_cadence', 'acf_sublag_fundamental', 'cosinor', 'acf_sublag_fundamental|cosinor',
        ]),
        'estimator__extractor__noop__spectral_stats': Categorical([
            'none',
            'peakedness|spectral_entropy',
            'peakedness|spectral_entropy|peak_period|parseval_ratio',
            'peakedness|spectral_entropy|peak_period|parseval_ratio|band_fractions|peak_neighbourhood_fraction',
        ]),
        'estimator__extractor__noop__quality': Categorical(['none', 'basic', 'report', 'features']),
        'estimator__extractor__noop__signal': Categorical(['raw', 'linear', 'anglez_rate', 'angular_speed', 'log_enmo', 'enmo', 'dyn_amp']),
        'estimator__extractor__noop__min_cycles': Integer(1, 5),
        'estimator__extractor__noop__min_samples': Categorical([16, 32, 48, 96, 192]),
        'estimator__extractor__noop__max_gap_samples': Categorical([2, 4, 8, 16, 32]),
        'estimator__extractor__noop__acf_harmonic_tolerance': Real(0.05, 0.35),
        'estimator__extractor__noop__detrend': Categorical(['none', 'constant', 'linear']),
        'estimator__extractor__noop__min_cadence_strength': Real(0.02, 0.35),
        'estimator__extractor__noop__power_branches': Categorical(['none', 'enmo', 'enmo|light', 'enmo|light|battery']),
        'estimator__extractor__include_tabular_engineered': Categorical([False, True]),
        'estimator__extractor__harmonize_instruments': Categorical([False, True]),
        'estimator__extractor__missing_feature_strategy': Categorical([
            'none', 'keep_nan', 'group_age_median', 'median', 'mean', 'constant_zero', 'knn', 'iterative',
        ]),
        'estimator__extractor__add_missing_indicators': Categorical([False, True]),
        'estimator__extractor__impute_knn_neighbors': Integer(2, 15),
        'estimator__extractor__impute_max_iter': Integer(3, 20),

        # ----------------------------------------------------------
        # 8. HANDCRAFTED ESTIMATOR — Domain Rules, Weights & Threshold Calibration
        # ----------------------------------------------------------
        'estimator__scoring_rule': Categorical(['autocorr_spectral_rule', 'weighted_z_index', 'multiplicative_hazard']),
        'estimator__w_internet_hours': Real(0.0, 2.2),
        'estimator__w_age': Real(0.0, 1.2),
        'estimator__w_age_internet': Real(0.0, 1.6),
        'estimator__w_sleep_disturbance': Real(0.0, 1.2),
        'estimator__w_inactivity': Real(0.0, 0.8),
        'estimator__w_cgas_deficit': Real(0.0, 0.8),
        'estimator__w_acf_irregularity': Real(0.0, 1.0),
        'estimator__w_sedentary_acf': Real(0.0, 1.0),
        'estimator__w_sri_irregularity': Real(0.0, 1.0),
        'estimator__w_nocturnal_ratio': Real(0.0, 0.9),
        'estimator__w_upright_night': Real(0.0, 0.9),
        'estimator__w_acrophase_delay': Real(0.0, 0.8),
        'estimator__w_period_drift': Real(0.0, 0.7),
        'estimator__w_stft_flux': Real(0.0, 0.7),
        'estimator__w_cwt_ultradian': Real(0.0, 0.7),
        'estimator__w_seglearn_kinematic': Real(0.0, 0.7),
        'estimator__w_cusum_prsa': Real(0.0, 0.6),
        'estimator__w_sleep_fragmentation': Real(0.0, 0.7),
        'estimator__w_lag_lead_coupling': Real(0.0, 0.7),
        'estimator__w_selected_static': Real(0.0, 0.8),
        'estimator__coupling_alpha': Real(0.0, 0.8),
        'estimator__threshold_calibration': Categorical(['optimize_qwk', 'quantile', 'fixed_midpoint', 'none']),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE = prepare_bayesian_space(BAYESIAN_PARAM_SPACE)


## Evolution

In [ ]:
# EVOLUTIONARY SEARCH ONLY
tsfel_minimal_evo = {
    'statistical': {
        'Mean': {'use': 'yes'},
        'Variance': {'use': 'yes'},
        'Max': {'use': 'yes'},
        'Min': {'use': 'yes'},
    }
}
tsfel_kinematic_evo = {
    'statistical': {
        'Mean': {'use': 'yes'},
        'Variance': {'use': 'yes'},
        'Root mean square': {'use': 'yes'},
    },
    'temporal': {
        'Peak to peak distance': {'use': 'yes'},
        'Zero crossing rate': {'use': 'yes'},
        'Slope': {'use': 'yes'},
    },
}

tsfel_entropy_evo = {
    'statistical': {
        'Entropy': {'use': 'yes'},
    },
    'temporal': {
        'Slope': {'use': 'yes'},
        'Zero crossing rate': {'use': 'yes'},
    },
    'spectral': {
        'Spectral entropy': {'use': 'yes'},
        'Fundamental frequency': {'use': 'yes'},
    },
}

EVOLUTIONARY_PARAM_SPACE = {
    # ----------------------------------------------------------
    # 1. AUGMENTOR (tsaug & Modality Dropout — 0.0 first so Seed Candidate 0 starts clean, up to extreme limits)
    # ----------------------------------------------------------
    'estimator__augmentor__prob': ECat([0.0, 0.25, 0.5, 0.75, 1.0]),
    'estimator__augmentor__per_aug_prob': ECat([0.0, 0.25, 0.35, 0.5, 0.75, 1.0]),
    'estimator__augmentor__augmentations': ECat([
        'none',
        None,
        'jitter|noise|scaling',
        'jitter|noise|scaling|magnitude_warp|time_warp',
        'time_warp|time_shift|crop|temporal_mask|channel_drop',
        'jitter|noise|scaling|magnitude_warp|time_warp|time_shift|crop|temporal_mask|channel_drop',
    ]),
    'estimator__augmentor__n_copies': ECat([0, 1, 2, 3, 4]),
    'estimator__augmentor__jitter_sigma': ECat([0.0, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0]),
    'estimator__augmentor__noise_std': ECat([0.0, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0]),
    'estimator__augmentor__scaling_sigma': ECat([0.0, 0.03, 0.05, 0.10, 0.25, 0.50, 1.0]),
    'estimator__augmentor__warp_sigma': ECat([0.0, 0.03, 0.05, 0.10, 0.25, 0.50, 1.0]),
    'estimator__augmentor__warp_num_knots': ECat([0, 2, 4, 6, 8, 12]),
    'estimator__augmentor__time_warp_max_speed_ratio': ECat([1.0, 1.15, 1.25, 1.5, 2.0, 3.0, 5.0]),
    'estimator__augmentor__time_warp_n_speed_change': ECat([0, 2, 3, 5, 8, 12, 20]),
    'estimator__augmentor__time_shift_frac': ECat([0.0, 0.02, 0.05, 0.10, 0.25, 0.50, 0.75, 0.95]),
    'estimator__augmentor__crop_frac_range': ECat([
        (1.0, 1.0),
        (0.85, 1.0),
        (0.75, 0.95),
        (0.5, 0.9),
        (0.4, 0.8),
        (0.1, 0.4),
    ]),
    'estimator__augmentor__temporal_num_masks': ECat([0, 1, 2, 4, 8, 15]),
    'estimator__augmentor__temporal_mask_frac': ECat([0.0, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90]),
    'estimator__augmentor__temporal_mask_fill': ECat(['none', 'ffill', 'bfill', 'mean', 0.0]),
    'estimator__augmentor__channel_drop_prob': ECat([0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75]),
    'estimator__augmentor__sensor_drop_prob': ECat([0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75]),
    'estimator__augmentor__sensor_drop_joint_prob': ECat([0.0, 0.05, 0.15, 0.20, 0.35, 0.50, 0.75]),
    'estimator__augmentor__random_state': ECat([random_state]),

    # ----------------------------------------------------------
    # 2. EXTRACTOR — Cleaning, Resampling, Gap Imputation & Sequence Quality
    # ----------------------------------------------------------
    'estimator__extractor__bin_minutes': ECat([5, 10, 15, 20, 30, 60]),
    'estimator__extractor__resample_method': ECat(['none', 'anti_alias_mean', 'poly', 'mean']),
    'estimator__extractor__non_wear_policy': ECat(['none', 'mask', 'impute_zero', 'impute_diurnal', 'keep_all']),
    'estimator__extractor__interp_mode': ECat([
        'none', 'bounded_linear', 'bounded_pchip', 'bounded_cubic', 'bounded_ffill', 'diurnal_median', 'zero',
    ]),
    'estimator__extractor__min_bin_wear_frac': ECat([0.0, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75]),
    'estimator__extractor__max_gap_bins': ECat([0, 1, 2, 4, 8, 12, 24, 48]),
    'estimator__extractor__impute_long_gaps': ECat([False, True]),
    'estimator__extractor__min_valid_wear_days': ECat([0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]),
    'estimator__extractor__maxlen_days': ECat([None, 3.0, 5.0, 7.0, 14.0, 21.0, 28.0]),
    'estimator__extractor__sequence_crop_mode': ECat(['none', 'pad_trunc', 'tsaug_crop', 'head', 'tail', 'center']),
    'estimator__extractor__window_cap_policy': ECat(['uniform', 'first', 'last']),
    'estimator__extractor__final_window_mode': ECat(['overlap', 'pad', 'drop']),
    'estimator__extractor__filter_problematic_sequences': ECat([False, True]),
    'estimator__extractor__ideal_skew_threshold': EFloat(0.5, 6.0),
    'estimator__extractor__problematic_features_threshold': EInt(1, 6),
    'estimator__extractor__filter_method': ECat(['none', 'iqr', 'zscore']),
    'estimator__extractor__filter_threshold': EFloat(1.5, 5.0),
    'estimator__extractor__max_outlier_features': EInt(1, 8),
    'estimator__extractor__clip_enmo': ECat([0.0, 1.5, 2.5, 4.0, 6.0, 10.0]),

    # ----------------------------------------------------------
    # 3. EXTRACTOR — Gravity Removal, Kalman Filtering & Kinematic Branches
    # ----------------------------------------------------------
    'estimator__extractor__linear_acc_mode': ECat(['none', 'rolling_mean', 'highpass_sos']),
    'estimator__extractor__baseline_window_bins': ECat([0, 4, 8, 16, 32, 64]),
    'estimator__extractor__highpass_hz': ECat([0.00005, 0.00010, 0.00015, 0.00020, 0.00025]),
    'estimator__extractor__motion_filter_mode': ECat(['none', 'kalman']),
    'estimator__extractor__kalman_process_noise': EFloat(1e-5, 5e-2, log=True),
    'estimator__extractor__kalman_measurement_noise': EFloat(1e-4, 1e-1, log=True),
    'estimator__extractor__use_dead_reckoning': ECat([False, True]),
    'estimator__extractor__dead_reckoning_detrend': ECat([False, True]),
    'estimator__extractor__acc_modes': ECat([
        'raw',
        'raw|smoothed|jerk',
        'raw|smoothed|jerk|velocity|linear|kalman',
        'raw|smoothed|jerk|velocity|freq_velocity|freq_displacement|linear|kalman',
    ]),
    'estimator__extractor__dc_offset_bins': EInt(1, 6),
    'estimator__extractor__smooth_window_bins': ECat([0, 2, 4, 8, 12, 16]),
    'estimator__extractor__smooth_alpha': ECat([None, 0.0, 0.15, 0.25, 0.5, 0.75]),
    'estimator__extractor__sedentary_threshold_g': ECat([0.010, 0.015, 0.020, 0.025, 0.035, 0.050]),

    # ----------------------------------------------------------
    # 4. EXTRACTOR — Autocorrelation, Lags & Leads, Circadian & Sleep Architecture
    # ----------------------------------------------------------
    'estimator__extractor__acf_detrend': ECat(['none', 'linear', 'constant']),
    'estimator__extractor__acf_bias': ECat(['unbiased', 'biased']),
    'estimator__extractor__short_lags_h': ECat([
        (0.25, 0.5, 1.0, 2.0),
        (0.5, 1.0, 2.0, 4.0),
        (0.25, 0.5, 1.0, 2.0, 4.0, 6.0),
        (1.0, 2.0, 4.0, 8.0, 12.0),
    ]),
    'estimator__extractor__period_band_hours': ECat([(16.0, 32.0), (18.0, 30.0), (20.0, 28.0), (22.0, 26.0)]),
    'estimator__extractor__fundamental_ratio': ECat([0.65, 0.75, 0.80, 0.85, 0.90, 0.95]),
    'estimator__extractor__min_acf_strength': ECat([0.0, 0.03, 0.05, 0.08, 0.12, 0.18]),
    'estimator__extractor__include_lags': ECat(['none', False, True]),
    'estimator__extractor__include_leads': ECat(['none', False, True]),
    'estimator__extractor__include_rolling': ECat(['none', False, True]),
    'estimator__extractor__lag_lead_hours': ECat([
        'none',
        (0.25, 0.5, 1.0),
        (0.5, 1.0, 2.0),
        (0.5, 1.0, 2.0, 4.0),
        (0.5, 1.0, 2.0, 4.0, 6.0, 12.0),
    ]),
    'estimator__extractor__rolling_window_hours': ECat([
        'none',
        (0.5, 1.5, 3.0),
        (1.0, 3.0),
        (1.0, 3.0, 6.0),
        (1.0, 3.0, 6.0, 12.0, 24.0),
    ]),
    'estimator__extractor__lag_lead_channels': ECat([
        'none',
        'log_enmo|light',
        'log_enmo|light|anglez',
        'log_enmo|enmo|dyn_amp|light|anglez|lin_enmo',
    ]),

    # ----------------------------------------------------------
    # 5. EXTRACTOR — Time-Frequency (STFT, Welch Moments, CWT Cone/Ridges, DWT)
    # ----------------------------------------------------------
    'estimator__extractor__stft_window_days': ECat([1.0, 1.5, 2.0, 3.0, 4.0]),
    'estimator__extractor__stft_overlap_ratio': ECat([0.0, 0.25, 0.5, 0.75, 0.85]),
    'estimator__extractor__stft_window_type': ECat(['none', 'hann', 'hamming', 'blackman']),
    'estimator__extractor__stft_scaling': ECat(['none', 'psd', 'spectrum']),
    'estimator__extractor__stft_detrend': ECat(['constant', 'linear', 'none']),
    'estimator__extractor__spectral_moment_max_peaks': EInt(2, 12),
    'estimator__extractor__cwt_wavelet': ECat(['none', 'morl', 'mexh', 'cmor1.5-1.0']),
    'estimator__extractor__cwt_n_scales': ECat([0, 8, 12, 16, 24, 32]),
    'estimator__extractor__cwt_min_period_h': ECat([1.0, 1.5, 2.0, 3.0, 4.0]),
    'estimator__extractor__cwt_max_period_h': ECat([24.0, 30.0, 36.0, 48.0, 72.0]),
    'estimator__extractor__dwt_wavelet': ECat(['none', 'db4', 'sym4', 'coif1', 'haar']),
    'estimator__extractor__dwt_level': ECat([0, 2, 3, 4, 5, 6]),

    # ----------------------------------------------------------
    # 6. EXTRACTOR — Open-Source Engines & Feature Selection
    # ----------------------------------------------------------
    'estimator__extractor__seglearn_window_days': ECat([0.5, 1.0, 2.0, 3.0, 4.0]),
    'estimator__extractor__seglearn_overlap': ECat([0.0, 0.25, 0.5, 0.75]),
    'estimator__extractor__seglearn_feature_set': ECat(['none', 'safe_all', 'base', 'emg_kinematic']),
    'estimator__extractor__catch22_catch24': ECat([False, True]),
    'estimator__extractor__catch22_channels': ECat([
        'none',
        'log_enmo',
        'log_enmo|dyn_amp',
        'log_enmo|dyn_amp|anglez',
        'log_enmo|dyn_amp|anglez|light',
    ]),
    'estimator__extractor__static_backend': ECat(['none', 'tsfel+tsfresh', 'tsfel', 'tsfresh']),
    'estimator__extractor__tsfel_domains': ECat([
        'none', 'statistical', 'temporal', 'spectral', 'statistical|temporal', 'statistical|temporal|spectral',
    ]),
    'estimator__extractor__tsfel_features': ECat([
        None,
        'none',
        json.dumps(tsfel_minimal_evo, sort_keys=True),
        json.dumps(tsfel_kinematic_evo, sort_keys=True),
        json.dumps(tsfel_entropy_evo, sort_keys=True),
    ]),
    'estimator__extractor__tsfresh_profile': ECat(['none', 'minimal', 'efficient']),
    'estimator__extractor__static_channels': ECat([
        'none',
        'log_enmo|dyn_amp',
        'log_enmo|dyn_amp|anglez|light',
        'log_enmo|dyn_amp|anglez|light|lin_enmo|enmo_kalman',
    ]),
    'estimator__extractor__tsflex_windows': ECat([
        'none', ('1h', '3h'), ('2h', '6h'), ('2h', '6h', '12h'), ('2h', '6h', '12h', '24h'),
    ]),
    'estimator__extractor__tsflex_strides': ECat(['30m', '1h', '2h', '4h', '6h']),
    'estimator__extractor__tsflex_channels': ECat([
        'none', 'log_enmo|dyn_amp', 'log_enmo|dyn_amp|light', 'log_enmo|dyn_amp|light|anglez',
    ]),
    'estimator__extractor__feature_selection_method': ECat(['none', 'correlation', 'f_classif']),
    'estimator__extractor__static_top_k': ECat([None, 8, 12, 24, 48, 96, 128]),
    'estimator__extractor__feature_selection_percentile': ECat([None, 10.0, 25.0, 50.0, 75.0, 90.0]),
    'estimator__extractor__feature_families': ECat([
        'none',
        'autocorr|lag_lead|circadian|sleep_arch|compliance',
        'autocorr|lag_lead|circadian|sleep_arch|motion|spectral|stft|wavelet|prsa_cusum|branches|compliance',
        'autocorr|lag_lead|circadian|sleep_arch|motion|spectral|stft|wavelet|seglearn|catch22|tsfel|tsfresh|tsflex|prsa_cusum|branches|compliance',
    ]),

    # ----------------------------------------------------------
    # 7. EXTRACTOR — NOOP Physiology & Tabular Missingness / Imputation
    # ----------------------------------------------------------
    'estimator__extractor__noop__motion_stats': ECat([
        'none',
        'enmo_ac_rms|motion_cv',
        'enmo_ac_rms|motion_cv|posture_var|jerk_floor',
        'enmo_ac_rms|motion_cv|posture_var|jerk_floor|dog_var|baevsky_si',
    ]),
    'estimator__extractor__noop__periodicity': ECat([
        'none', 'acf', 'acf_harmonic_checked', 'acf_cadence', 'acf_sublag_fundamental', 'cosinor', 'acf_sublag_fundamental|cosinor',
    ]),
    'estimator__extractor__noop__spectral_stats': ECat([
        'none',
        'peakedness|spectral_entropy',
        'peakedness|spectral_entropy|peak_period|parseval_ratio',
        'peakedness|spectral_entropy|peak_period|parseval_ratio|band_fractions|peak_neighbourhood_fraction',
    ]),
    'estimator__extractor__noop__quality': ECat(['none', 'basic', 'report', 'features']),
    'estimator__extractor__noop__signal': ECat(['raw', 'linear', 'anglez_rate', 'angular_speed', 'log_enmo', 'enmo', 'dyn_amp']),
    'estimator__extractor__noop__min_cycles': EInt(1, 5),
    'estimator__extractor__noop__min_samples': ECat([16, 32, 48, 96, 192]),
    'estimator__extractor__noop__max_gap_samples': ECat([2, 4, 8, 16, 32]),
    'estimator__extractor__noop__acf_harmonic_tolerance': EFloat(0.05, 0.35),
    'estimator__extractor__noop__detrend': ECat(['none', 'constant', 'linear']),
    'estimator__extractor__noop__min_cadence_strength': EFloat(0.02, 0.35),
    'estimator__extractor__noop__power_branches': ECat(['none', 'enmo', 'enmo|light', 'enmo|light|battery']),
    'estimator__extractor__include_tabular_engineered': ECat([False, True]),
    'estimator__extractor__harmonize_instruments': ECat([False, True]),
    'estimator__extractor__missing_feature_strategy': ECat([
        'none', 'keep_nan', 'group_age_median', 'median', 'mean', 'constant_zero', 'knn', 'iterative',
    ]),
    'estimator__extractor__add_missing_indicators': ECat([False, True]),
    'estimator__extractor__impute_knn_neighbors': EInt(2, 15),
    'estimator__extractor__impute_max_iter': EInt(3, 20),

    # ----------------------------------------------------------
    # 8. HANDCRAFTED ESTIMATOR — Domain Rules, Weights & Threshold Calibration
    # ----------------------------------------------------------
    'estimator__scoring_rule': ECat(['autocorr_spectral_rule', 'weighted_z_index', 'multiplicative_hazard']),
    'estimator__w_internet_hours': EFloat(0.0, 2.2),
    'estimator__w_age': EFloat(0.0, 1.2),
    'estimator__w_age_internet': EFloat(0.0, 1.6),
    'estimator__w_sleep_disturbance': EFloat(0.0, 1.2),
    'estimator__w_inactivity': EFloat(0.0, 0.8),
    'estimator__w_cgas_deficit': EFloat(0.0, 0.8),
    'estimator__w_acf_irregularity': EFloat(0.0, 1.0),
    'estimator__w_sedentary_acf': EFloat(0.0, 1.0),
    'estimator__w_sri_irregularity': EFloat(0.0, 1.0),
    'estimator__w_nocturnal_ratio': EFloat(0.0, 0.9),
    'estimator__w_upright_night': EFloat(0.0, 0.9),
    'estimator__w_acrophase_delay': EFloat(0.0, 0.8),
    'estimator__w_period_drift': EFloat(0.0, 0.7),
    'estimator__w_stft_flux': EFloat(0.0, 0.7),
    'estimator__w_cwt_ultradian': EFloat(0.0, 0.7),
    'estimator__w_seglearn_kinematic': EFloat(0.0, 0.7),
    'estimator__w_cusum_prsa': EFloat(0.0, 0.6),
    'estimator__w_sleep_fragmentation': EFloat(0.0, 0.7),
    'estimator__w_lag_lead_coupling': EFloat(0.0, 0.7),
    'estimator__w_selected_static': EFloat(0.0, 0.8),
    'estimator__coupling_alpha': EFloat(0.0, 0.8),
    'estimator__threshold_calibration': ECat(['optimize_qwk', 'quantile', 'fixed_midpoint', 'none']),
    'estimator__random_state': ECat([random_state]),
}
EVOLUTIONARY_SEARCH_SPACE = prepare_evolutionary_space(EVOLUTIONARY_PARAM_SPACE)

for space_name, candidate_space in (
    ('grid', GRID_PARAM_SPACE),
    ('bayesian', BAYESIAN_PARAM_SPACE),
    ('evolutionary', EVOLUTIONARY_SEARCH_SPACE),
):
    unknown_params = set(candidate_space) - set(pipeline.get_params(deep=True))
    if unknown_params:
        raise ValueError(f'{space_name}: unknown={sorted(unknown_params)}')

if search_mode == 'bayes':
    param_space = BAYESIAN_PARAM_SPACE
elif search_mode == 'evo':
    param_space = EVOLUTIONARY_SEARCH_SPACE
else:
    param_space = GRID_PARAM_SPACE
print(f'Validated {len(GRID_PARAM_SPACE)} Grid, {len(BAYESIAN_PARAM_SPACE)} Bayesian, and {len(EVOLUTIONARY_SEARCH_SPACE)} Evolutionary parameters.')


# Train

In [ ]:
if search_mode == 'bayes':
    if not SKOPT_AVAILABLE:
        raise ImportError('scikit-optimize is required for Bayesian search')
    search = HierarchicalBayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=candidate_score,
        cv=cv_object,
        refit=refit,
        random_state=random_state,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )
elif search_mode == 'evo':
    search = HierarchicalGASearchCV(
        estimator=pipeline,
        param_distributions=param_space,
        population_size=population_size,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        scoring=candidate_score,
        cv=cv_object,
        refit=refit,
        random_state=random_state,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )
else:
    deduped_grid = deduplicate_param_grid(param_space, base_params=pipeline.get_params(deep=True))
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=deduped_grid,
        scoring={'accuracy_pct': candidate_accuracy_pct, 'qwk': candidate_score},
        cv=cv_object,
        refit=refit,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )

validation_runs = run_validation_searches(
    search=search,
    experiment_df=experiment_df,
    target_col=TARGET_COL,
    validation_param_space=VALIDATION_PARAM_SPACE,
    train_size=train_size,
    n_splits=n_splits,
    random_state=random_state,
    results_dir=results_dir,
    experiment_name=experiment_name,
    search_mode=search_mode,
    timestamp=timestamp,
    save_results=save_results,
    keep_history=keep_history,
)

last_run = validation_runs['last_run']
protocol_results = validation_runs['protocol_results']
experiment_succeeded = bool(last_run.get('experiment_succeeded', False))
if experiment_succeeded:
    search = last_run['search']
    X_train = last_run['X_train']
    y_train = last_run['y_train']
    X_test = last_run['X_test']
    y_test = last_run['y_test']
    eval_results = last_run['holdout_results']


# Analysis

In [ ]:
if experiment_succeeded:
    print('Last validation protocol:', last_run['validation_mode'])
    print('Holdout normal score (%):', eval_results.get('accuracy_pct'))
    print('Holdout competition score (QWK):', eval_results['competition_score'])
pd.DataFrame(protocol_results)


In [ ]:
print('Experiments CSV:', validation_runs['experiments_path'])
print('Summary JSON:', validation_runs['summary_path'])
candidates_df = validation_runs.get('candidates_df', build_quick_results_df(search) if experiment_succeeded else pd.DataFrame())
print(f'Candidate results table: {len(candidates_df)} rows x {candidates_df.shape[1]} columns (all parameters expanded into columns)')
candidates_df.head(10)


In [ ]:
if experiment_succeeded:
    test_df = load_cmi_dataset(data_root=data_root, split='test', dropna_target=False)
    sample_sub_path = data_root / 'sample_submission.csv'
    sub_workspace_path = (Path(workspace_root) / 'submission.csv') if save_results else None
    submission_df = export_submission(
        search.best_estimator_,
        test_df,
        output_path=sub_workspace_path,
        sample_sub_path=sample_sub_path if sample_sub_path.exists() else None,
    )
    print('Submission written to:', sub_workspace_path)
    display(submission_df.head(10))


# Trained-model permutation importance

In [ ]:
importance_df = pd.DataFrame()
if experiment_succeeded and run_permutation_importance:
    importance_dir = (results_dir / 'importance') if save_results and results_dir is not None else None
    importance_df = compute_permutation_importance(
        search.best_estimator_,
        X_train,
        y_train,
        repeats=3,
        random_state=random_state,
        include_noop_individual=True,
        require_internal_validation=True,
        output_dir=importance_dir,
        prefix=f'{timestamp}_{last_run["validation_mode"]}_permutation_importance',
    )
    print(
        'Permutation importance scope:',
        importance_df.attrs.get('scope'),
        '| baseline QWK:',
        round(float(importance_df.attrs.get('baseline_qwk', float('nan'))), 5),
        '| baseline macro-F1:',
        round(float(importance_df.attrs.get('baseline_macro_f1', float('nan'))), 5),
    )
    if importance_df.attrs.get('paths'):
        print('Saved importance artifacts:', importance_df.attrs['paths'])
importance_df.head(25)
